# CLI

> Main command-line interface to the library exposing the functionality
of elastic library for direct use - without writing any python code.

In [1]:
#| default_exp cli

In [2]:
#| hide
from nbdev.showdoc import *
from click.testing import CliRunner

In [3]:
import click
import ase.io
import elastic
from click import echo
from glob import glob

In [4]:
#| hide
#| exporti
_version_message=("Elastic, version %(version)s\n"
                  '(C) 2011-2025 by Paweł T. Jochym\n'
                  '    License: GPL v3 or later')

In [5]:
#| hide
#| exporti
def run_cli_cmd(cmd, args, prt_result=False):
    print(f'$ {cmd.name} {args}\n')
    run = CliRunner().invoke(cmd, args) 
    print(run.output)
    if prt_result or run.exit_code!=0:
        print(run)
        if run.exit_code!=0:
            import traceback
            traceback.print_tb(run.exc_info[-1])        

In [6]:
#| exporti
def banner(verbose):
    if verbose > 1:
        import elastic
        echo('Elastic ver. %s\n----------------------' %
             elastic.__version__)

In [7]:
#| exporti
def process_calc(fn):
    from time import sleep
    sleep(1)

In [8]:
#| hide
#| exporti

@click.group()
@click.option('--vasp', 'frmt', flag_value='vasp',
              help='Use VASP formats (default)', default=True)
@click.option('--abinit', 'frmt', flag_value='abinit',
              help='Use AbInit formats')
@click.option('--aims', 'frmt', flag_value='aims',
              help='Use FHI-aims formats')
@click.option('--cij', 'action', flag_value='cij',
              help='Generate deformations for Cij (default)', default=True)
@click.option('--eos', 'action', flag_value='eos',
              help='Generate deformations for Equation of State')
@click.option('-v', '--verbose', count=True, help='Increase verbosity')
@click.version_option(elastic.__version__, '-V', '--version', message=_version_message)
@click.pass_context
def elastic(ctx, frmt, action, verbose):
    '''Command-line interface to the elastic library.'''

    banner(ctx.params['verbose'])

In [9]:
#|echo: false
run_cli_cmd(elastic, '-V')
run_cli_cmd(elastic, '--help')

$ elastic -V

Elastic, version 6.0.0
(C) 2011-2025 by Paweł T. Jochym
    License: GPL v3 or later

$ elastic --help

Usage: elastic [OPTIONS] COMMAND [ARGS]...

  Command-line interface to the elastic library.

Options:
  --vasp         Use VASP formats (default)
  --abinit       Use AbInit formats
  --aims         Use FHI-aims formats
  --cij          Generate deformations for Cij (default)
  --eos          Generate deformations for Equation of State
  -v, --verbose  Increase verbosity
  -V, --version  Show the version and exit.
  --help         Show this message and exit.



In [10]:
#| hide
#| exporti

@elastic.command()
@click.option('-n', '--num', 'num', default=5, type=int,
              help='Number of generated deformations per axis (default: 5)')
@click.option('-l', '--lo', 'lo', default=0.98, type=float,
              help='Lower relative volume for EOS scan (default: 0.98)')
@click.option('-h', '--hi', 'hi', default=1.02, type=float,
              help='Upper relative volume for EOS scan (default: 1.02)')
@click.option('-s', '--size', 'size', default=2.0, type=float,
              help='Deformation size for Cij scan (% or deg., default: 2.0)')
@click.argument('struct', type=click.Path(exists=True))
@click.pass_context
def gen(ctx, num, lo, hi, size, struct):
    '''Generate deformed structures'''

    verbose = ctx.parent.params['verbose']
    frmt = ctx.parent.params['frmt']
    action = ctx.parent.params['action']
    cryst = ase.io.read(struct, format=frmt)
    fn_tmpl = action
    if frmt == 'vasp':
        fn_tmpl += '_%03d.POSCAR'
        kwargs = {'vasp5': True, 'direct': True}
    elif frmt == 'abinit':
        fn_tmpl += '_%03d.abinit'
        kwargs = {}
    elif frmt == "aims":
        fn_tmpl += '_%03d.in'
        kwargs = {}

    if verbose:
        from elastic import get_lattice_type
        nr, brav, sg, sgn = get_lattice_type(cryst)
        echo('%s lattice (%s): %s' % (brav, sg, cryst.get_chemical_formula()))
        if action == 'cij':
            echo('Generating {:d} deformations of {:.1f}(%/degs.) per axis'.format(
                    num, size))
        elif action == 'eos':
            echo('Generating {:d} deformations from {:.3f} to {:.3f} of V0'.format(
                    num, lo, hi))

    if action == 'cij':
        from elastic import get_elementary_deformations
        systems = get_elementary_deformations(cryst, n=num, d=size)
    elif action == 'eos':
        from elastic import scan_volumes
        systems = scan_volumes(cryst, n=num, lo=lo, hi=hi)

    systems.insert(0, cryst)
    if verbose:
        echo(f'Writing {len(systems)} deformation files ({fn_tmpl}).')
    for n, s in enumerate(systems):
        ase.io.write(fn_tmpl % n, s, format=frmt, **kwargs)

In [11]:
#|echo: false
run_cli_cmd(elastic,
            'gen '
            '--help')

$ elastic gen --help

Usage: elastic gen [OPTIONS] STRUCT

  Generate deformed structures

Options:
  -n, --num INTEGER  Number of generated deformations per axis (default: 5)
  -l, --lo FLOAT     Lower relative volume for EOS scan (default: 0.98)
  -h, --hi FLOAT     Upper relative volume for EOS scan (default: 1.02)
  -s, --size FLOAT   Deformation size for Cij scan (% or deg., default: 2.0)
  --help             Show this message and exit.



In [12]:
#|echo: false
run_cli_cmd(elastic,
            '--eos -v '
            'gen '
            'data/POSCAR')

$ elastic --eos -v gen data/POSCAR

Cubic lattice (Fm-3m): Mg4O4
Generating 5 deformations from 0.980 to 1.020 of V0
Writing 6 deformation files (eos_%03d.POSCAR).



In [13]:
#|echo: false
run_cli_cmd(elastic,
            '--cij -v '
            'gen '
            'data/POSCAR')

$ elastic --cij -v gen data/POSCAR

Cubic lattice (Fm-3m): Mg4O4
Generating 5 deformations of 2.0(%/degs.) per axis
Writing 11 deformation files (cij_%03d.POSCAR).



In [14]:
#| hide
#| exporti

@elastic.command()
@click.argument('files', type=click.Path(exists=True), nargs=-1)
@click.pass_context
def proc(ctx, files):
    '''Process calculated structures'''

    def calc_reader(fn, verb):
        if verb>1:
            echo('Reading: {:<60s}\r'.format(fn), nl=False, err=True)
        return ase.io.read(fn)

    verbose = ctx.parent.params['verbose']
    action = ctx.parent.params['action']
    systems = [calc_reader(calc, verbose) for calc in files]
    if verbose :
        echo('', err=True)
    if action == 'cij':
        import elastic
        cij = elastic.get_elastic_tensor(systems[0], systems=systems[1:])
        msv = cij[1][3].max()
        eps = 1e-4
        if verbose:
            echo('Cij solution\n'+30*'-')
            echo(' Solution rank: {:2d}{}'.format(
                    cij[1][2],
                    ' (undetermined)' if cij[1][2] < len(cij[0]) else ''))
            if cij[1][2] == len(cij[0]):
                echo(' Square of residuals: {:7.2g}'.format(cij[1][1]))
            echo(' Relative singular values:')
            for sv in cij[1][3]/msv:
                echo('{:7.4f}{}'.format(
                        sv, '* ' if (sv) < eps else '  '), nl=False)
            echo('\n\nElastic tensor (GPa):')
            for dsc in elastic.get_cij_order(systems[0]):
                echo('{: >7s}  '.format(dsc), nl=False)
            echo('\n'+30*'-')
        for c, sv in zip(cij[0], cij[1][3]/msv):
            echo('{:7.2f}{}'.format(
                    c/ase.units.GPa, '* ' if sv < eps else '  '), nl=False)
        echo()
    elif action == 'eos':
        import elastic
        eos, pvdat = elastic.get_EOS(systems[0], systems=systems[1:])
        eos[1] /= ase.units.GPa
        if verbose:
            echo('# %7s (A^3)%7s (GPa) %7s' % ("V0", "K", "K'"))
        echo('     %7.2f        %7.2f    %7.2f' % tuple(eos))

In [15]:
#|echo: false
run_cli_cmd(elastic,
            'proc '
            '--help')

$ elastic proc --help

Usage: elastic proc [OPTIONS] [FILES]...

  Process calculated structures

Options:
  --help  Show this message and exit.



In [16]:
#|echo: false
run_cli_cmd(elastic,
            '--eos -v -v proc ' + 
            ' '.join(sorted(glob('data/calc-eos_*/vasprun.xml')))
           )

$ elastic --eos -v -v proc data/calc-eos_000/vasprun.xml data/calc-eos_001/vasprun.xml data/calc-eos_002/vasprun.xml data/calc-eos_003/vasprun.xml data/calc-eos_004/vasprun.xml data/calc-eos_005/vasprun.xml

Elastic ver. 6.0.0
----------------------
Reading: data/calc-eos_005/vasprun.xml                               
#      V0 (A^3)      K (GPa)      K'
       74.24         165.68       2.11



In [19]:
#|echo: false
run_cli_cmd(elastic,
            '--cij -v proc ' + 
            ' '.join(sorted(glob('data/calc-cij_*/vasprun.xml')))
           )

$ elastic --cij -v proc data/calc-cij_000/vasprun.xml data/calc-cij_001/vasprun.xml data/calc-cij_002/vasprun.xml data/calc-cij_003/vasprun.xml data/calc-cij_004/vasprun.xml data/calc-cij_005/vasprun.xml data/calc-cij_006/vasprun.xml data/calc-cij_007/vasprun.xml data/calc-cij_008/vasprun.xml data/calc-cij_009/vasprun.xml data/calc-cij_010/vasprun.xml


Cij solution
------------------------------
 Solution rank:  3
 Square of residuals: 0.00049
 Relative singular values:
 1.0000   0.7071   0.6354  

Elastic tensor (GPa):
   C_11     C_12     C_44  
------------------------------
 321.15    95.88   143.44  

